In [6]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import numpy as np

# Load the Titanic dataset
try:
    data = pd.read_csv("/tested.csv")
    print("Original Data Head:")
    print(data.head())

    # Define features (X) and target (y)
    # Assuming 'Survived' is the target variable in the test set as well,
    # although in the actual competition 'Survived' is in the training set.
    # If 'Survived' is not in 'tested.csv', this will cause an error.
    # In a real scenario, you'd train on 'train.csv' and predict on 'test.csv'.
    # For this modification, assuming 'Survived' is present for demonstration.
    if 'Survived' in data.columns:
        y = data['Survived']
        x = data.drop('Survived', axis=1)
    else:
        # If 'Survived' is not in the test set, we'll just use all other columns as features
        # and skip the target variable split and model training/evaluation parts that require y.
        # However, the user asked to include all columns, so we'll proceed assuming 'Survived' is there
        # or that the user intends to use this for prediction after training on a different set.
        # Let's assume 'Survived' is the implied target even if not present for now,
        # but we'll use all columns for X as requested.
        x = data.copy()
        y = None # No target variable in this case

    print("\nFeatures (X) Head:")
    print(x.head())
    if y is not None:
        print("\nTarget (y) Head:")
        print(y.head())

    # Identify categorical and numerical columns
    categorical_features = ['Sex', 'Embarked']
    numerical_features = ['Age', 'SibSp', 'Parch', 'Fare'] # Add other numerical columns if needed

    # Create preprocessing pipelines for numerical and categorical features
    numerical_transformer = SimpleImputer(strategy='median') # Impute missing numerical values
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')), # Impute missing categorical values
        ('onehot', OneHotEncoder(handle_unknown='ignore'))    # One-hot encode categorical features
    ])

    # Create a column transformer to apply different transformations to different columns
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features)])

    # Create a pipeline that first preprocesses the data and then applies Logistic Regression
    model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                     ('classifier', LogisticRegression(max_iter=1000))])


    if y is not None:
        # Split data into training and testing sets
        # Note: Splitting the test set further is generally not done in a real Kaggle scenario.
        # You would typically train on train.csv and predict on test.csv.
        # This split is for demonstration purposes if you want to evaluate performance on a subset of tested.csv.
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

        # Train the model
        model_pipeline.fit(x_train, y_train)

        # Make predictions
        y_pred = model_pipeline.predict(x_test)

        # Evaluate the model
        accuracy = accuracy_score(y_test, y_pred)
        print("\nModel Accuracy:", accuracy)
    else:
        print("\n'Survived' column not found in the dataset. Cannot train or evaluate the model.")
        print("Preprocessing the data using the defined pipeline:")
        x_processed = model_pipeline.named_steps['preprocessor'].fit_transform(x)
        print("Processed data shape:", x_processed.shape)
        # You would typically load a trained model here and use it to predict on x_processed


except FileNotFoundError:
    print("Error: 'tested.csv' not found. Please upload the file or provide the correct path.")
except KeyError as e:
    print(f"Error: Column {e} not found in the dataset. Please check the column names.")
except Exception as e:
    print(f"An error occurred: {e}")

Original Data Head:
   PassengerId  Survived  Pclass  \
0          892         0       3   
1          893         1       3   
2          894         0       2   
3          895         0       3   
4          896         1       3   

                                           Name     Sex   Age  SibSp  Parch  \
0                              Kelly, Mr. James    male  34.5      0      0   
1              Wilkes, Mrs. James (Ellen Needs)  female  47.0      1      0   
2                     Myles, Mr. Thomas Francis    male  62.0      0      0   
3                              Wirz, Mr. Albert    male  27.0      0      0   
4  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female  22.0      1      1   

    Ticket     Fare Cabin Embarked  
0   330911   7.8292   NaN        Q  
1   363272   7.0000   NaN        S  
2   240276   9.6875   NaN        Q  
3   315154   8.6625   NaN        S  
4  3101298  12.2875   NaN        S  

Features (X) Head:
   PassengerId  Pclass                        